# 05. 회귀 모델 비교와 하이퍼파라미터 튜닝
여러 회귀 모델을 같은 데이터 분할 기준으로 비교하고, GridSearchCV로 튜닝한 뒤 검증 성능을 기준으로 모델을 비교합니다.

> 데이터 확인·분할·스케일링은 이미 학습했으므로 준비 단계는 크게 묶고, **모델 비교와 튜닝**에 집중합니다.

In [ ]:
from common.libraries import *

# 이번 학습에서 집중할 기능
# - 여러 회귀 모델을 동일 조건으로 비교
# - GridSearchCV : 교차검증으로 하이퍼파라미터 탐색
# - 검증 R²를 기준으로 비교하고 테스트 데이터는 최종 확인용으로 사용
# - joblib / pickle : 학습 모델 저장 및 불러오기

In [ ]:
# 기본 준비: 데이터 로드 → X/y → 6:2:2 분할 → 표준화
df = pd.read_csv("./data/housing_price_1000.csv")

X = df[["면적", "층수", "역까지_도보시간", "주변_편의시설_점수"]]
y = df["주택_가격"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train / Val / Test :", X_train.shape, X_val.shape, X_test.shape)

## 기본 모델 비교
`score()`는 회귀 모델에서 R²를 반환합니다.  
훈련 점수와 검증 점수의 차이도 함께 확인해 과적합 여부를 봅니다.

In [ ]:
models = [
    LinearRegression(),
    Ridge(),
    Lasso(),
    RandomForestRegressor(random_state=42),
    ExtraTreesRegressor(random_state=42),
    GradientBoostingRegressor(random_state=42),
    HistGradientBoostingRegressor(random_state=42),
    XGBRegressor(random_state=42, n_jobs=-1),
]

results = []

for model in models:
    model_name = model.__class__.__name__
    model.fit(X_train_scaled, y_train)

    train_r2 = model.score(X_train_scaled, y_train)
    val_r2 = model.score(X_val_scaled, y_val)
    test_r2 = model.score(X_test_scaled, y_test)

    results.append([
        model_name,
        train_r2,
        val_r2,
        test_r2,
        train_r2 - val_r2,
    ])

result_df = pd.DataFrame(
    results,
    columns=["모델명", "훈련-R²", "검증-R²", "테스트-R²", "훈련-검증차이"]
)
result_df["랭킹"] = result_df["검증-R²"].rank(method="min", ascending=False).astype(int)
result_df.sort_values(["랭킹", "훈련-검증차이"], key=lambda s: s.abs() if s.name == "훈련-검증차이" else s)

## GridSearchCV로 튜닝
모델마다 지원하는 하이퍼파라미터가 다르므로 파라미터 표를 따로 정의합니다.  
`LinearRegression`은 별도의 주요 튜닝 파라미터가 없어 기본 비교 결과를 그대로 사용합니다.

In [ ]:
tuning_targets = {
    "Ridge": (
        Ridge(),
        {"alpha": [0.01, 0.1, 1, 10, 100]}
    ),
    "Lasso": (
        Lasso(max_iter=10000),
        {"alpha": [0.001, 0.01, 0.1, 1, 10]}
    ),
    "RandomForestRegressor": (
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {
            "n_estimators": [50, 100, 150],
            "max_depth": [None, 10],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2, 4],
        }
    ),
    "ExtraTreesRegressor": (
        ExtraTreesRegressor(random_state=42, n_jobs=-1),
        {
            "n_estimators": [50, 100, 150],
            "max_depth": [None, 10],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2, 4],
        }
    ),
    "GradientBoostingRegressor": (
        GradientBoostingRegressor(random_state=42),
        {
            "n_estimators": [50, 100, 150],
            "max_depth": [2, 3, 5],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2, 4],
        }
    ),
    "HistGradientBoostingRegressor": (
        HistGradientBoostingRegressor(random_state=42),
        {
            "max_iter": [50, 100, 150],
            "max_depth": [None, 10],
            "min_samples_leaf": [5, 10, 20],
        }
    ),
    "XGBRegressor": (
        XGBRegressor(random_state=42, n_jobs=-1),
        {
            "n_estimators": [50, 100, 150],
            "max_depth": [3, 6, 10],
            "min_child_weight": [1, 2, 4],
        }
    ),
}

In [ ]:
tuned_results = []
best_models = {}

for model_name, (model, params) in tuning_targets.items():
    grid = GridSearchCV(
        estimator=model,
        param_grid=params,
        scoring="r2",
        cv=5,
        n_jobs=-1,
    )
    grid.fit(X_train_scaled, y_train)

    best_model = grid.best_estimator_
    best_models[model_name] = best_model

    train_pred = best_model.predict(X_train_scaled)
    val_pred = best_model.predict(X_val_scaled)
    test_pred = best_model.predict(X_test_scaled)

    train_r2 = r2_score(y_train, train_pred)
    val_r2 = r2_score(y_val, val_pred)
    test_r2 = r2_score(y_test, test_pred)

    tuned_results.append([
        model_name,
        grid.best_params_,
        train_r2,
        val_r2,
        test_r2,
        train_r2 - val_r2,
    ])

tuned_df = pd.DataFrame(
    tuned_results,
    columns=["모델명", "경우의수", "훈련-R²", "검증-R²", "테스트-R²", "훈련-검증차이"]
)

tuned_df["랭킹"] = tuned_df["검증-R²"].rank(method="min", ascending=False).astype(int)
tuned_df.sort_values(["랭킹", "훈련-검증차이"], key=lambda s: s.abs() if s.name == "훈련-검증차이" else s)

## 최종 모델 확인
모델 선택은 **검증 R²를 우선**하고, 비슷한 경우 훈련-검증 차이가 작은 모델을 함께 봅니다.  
테스트 점수는 선택 과정에 반복해서 사용하기보다 마지막 성능 확인에 사용하는 것이 좋습니다.

In [ ]:
best_row = tuned_df.sort_values(
    ["검증-R²", "훈련-검증차이"],
    ascending=[False, True],
    key=lambda s: s.abs() if s.name == "훈련-검증차이" else s
).iloc[0]

best_name = best_row["모델명"]
best_model = best_models[best_name]

print("최종 모델 :", best_name)
print("검증 R² :", best_row["검증-R²"])
print("테스트 R² :", best_row["테스트-R²"])
print("훈련-검증 차이 :", best_row["훈련-검증차이"])

## 모델 저장과 새 데이터 예측

In [ ]:
# 모델과 스케일러를 함께 저장
joblib.dump(best_model, "./models/best_regression_model.pkl")
joblib.dump(scaler, "./models/regression_scaler.pkl")

# 새 데이터는 학습 때 사용한 scaler로 transform만 수행
home_data = [[69.04, 27, 21.4, 2.7]]
home_scaled = scaler.transform(home_data)
prediction = best_model.predict(home_scaled)

print("예측값 :", prediction[0])